In [ ]:
import arviz as az
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import Markdown

from emu_renewal.outputs import get_output_dir, melt_df_except_first_level, get_multianalysis_dispvals_from_idatas, get_multianalysis_procvals_from_idatas
from emu_renewal.outputs import get_multianalysis_ind_spaghetti, get_multianalysis_procvals
from emu_renewal.plotting import plot_process_comparison, plot_updates_comparison, plot_mob_update_comparison

set_matplotlib_formats("svg")

In [ ]:
country = "France"
analysis_times = {
    "no_mob": "20250121_1108",
    "google_nonresi_linear": "20250121_1036",
}

In [ ]:
# Can compare more than two runs with the following functions
alternative_times = {
    "google_nonresi_square": "20250121_1046",
    "fb_linear": "20250121_1010",
    "fb_square": "20250121_1028",
}
all_analysis_times = analysis_times | alternative_times

In [ ]:
#| fig-cap: "Variable process residual function with inclusion of various mobility modifiers."
spagh_df = get_multianalysis_ind_spaghetti(country, "process", all_analysis_times)
fig = plot_process_comparison(spagh_df, all_analysis_times.keys(), ["k", "cornflowerblue", "mediumblue", "red", "darkred"], 0.1);
fig.legend(bbox_to_anchor=[0.85, 0.75], loc="right")

In [ ]:
#| fig-cap: "Variable process update values with inclusion of various mobility modifiers."
idatas = {k: az.from_netcdf(get_output_dir(country, k, v) / "idata.nc") for k, v in all_analysis_times.items()}
plot_mob_update_comparison(idatas, 0.3, fig_height=14)

In [ ]:
multianalysis_proc_df = get_multianalysis_procvals_from_idatas(idatas)
multianalysis_disp_df = get_multianalysis_dispvals_from_idatas(idatas)

In [ ]:
# | fig-cap: "Dispersion parameter posteriors with inclusion of various mobility modifiers. Lower values for the dispersion parameter allow for smaller updates to the variable process over time."
fig, ax = plt.subplots(figsize=(6, 6))
disp_comparison_df = melt_df_except_first_level(multianalysis_disp_df)
sns.kdeplot(disp_comparison_df, fill=True, ax=ax)
ax.set_xlim([0.0, 0.25]);

In [ ]:
#| tbl-cap: Median 
medians = disp_comparison_df.median()
medians.index.name = "Analysis type"
medians.name = "Median values"
Markdown(medians.to_markdown())